In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from multiprocessing import Pool, cpu_count
import time

# 加载评分函数
from cnot_utils import evaluate_pulse

# 设置中文字体支持
rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
rcParams['axes.unicode_minus'] = False


In [ ]:

# 运行次数
iters_num = 100

# 加载脉冲
pulses_1 = np.load('results/pulses_spsa.npy')  # Shape: (30, 2)

phi = 0.0

print(f"开始并行评分 {iters_num} 次...")
print(f"使用 CPU 核心数: {cpu_count()}")

# 准备并行计算参数
args_list_1 = [(pulses_1, phi, False) for _ in range(iters_num)]


# 使用多进程并行计算
start_time = time.time()
with Pool() as pool:
    # 并行计算两组脉冲的评分
    results_1 = pool.map(evaluate_pulse, args_list_1)

end_time = time.time()

# 提取结果
score_list_1 = [r[0] for r in results_1]
gate_error_list_1 = [r[1] for r in results_1]
gate_fidelity_list_1 = [r[2] for r in results_1]
leakage_list_1 = [r[3] for r in results_1]
penalty_list_1 = [r[4] for r in results_1]

print(f"并行计算耗时: {end_time - start_time:.2f} 秒")

# 打印统计信息
print("\n=== pulses_spsa 统计信息 ===")
print(f"平均评分: {np.mean(score_list_1):.6f} ± {np.std(score_list_1):.6f}")
print(f"门错误: {np.mean(gate_error_list_1):.6f} ± {np.std(gate_error_list_1):.6f}")
print(f"门保真度: {np.mean(gate_fidelity_list_1):.6f} ± {np.std(gate_fidelity_list_1):.6f}")
print(f"泄漏: {np.mean(leakage_list_1):.6f} ± {np.std(leakage_list_1):.6f}")
print(f"惩罚: {np.mean(penalty_list_1):.6f} ± {np.std(penalty_list_1):.6f}")


In [ ]:

# 绘制详细结果
plt.figure(figsize=(15, 12))

# 子图1: 评分对比
plt.subplot(2, 2, 1)
plt.plot(range(1, iters_num+1), score_list_1, 'o-', label='pulses_spsa', linewidth=2, markersize=8)
plt.xlabel('运行次数')
plt.ylabel('总体评分')
plt.title('不同脉冲文件的评分对比')
plt.legend()
plt.grid(True, alpha=0.3)

# 子图2: 门保真度对比
plt.subplot(2, 2, 2)
plt.plot(range(1, iters_num+1), gate_fidelity_list_1, 'o-', label='pulses_spsa', linewidth=2, markersize=8)
plt.xlabel('运行次数')
plt.ylabel('门保真度')
plt.title('门保真度对比')
plt.legend()
plt.grid(True, alpha=0.3)

# 子图3: 泄漏对比
plt.subplot(2, 2, 3)
plt.plot(range(1, iters_num+1), leakage_list_1, 'o-', label='pulses_spsa', linewidth=2, markersize=8)
plt.xlabel('运行次数')
plt.ylabel('泄漏')
plt.title('泄漏对比')
plt.legend()
plt.grid(True, alpha=0.3)

# 子图4: 惩罚对比
plt.subplot(2, 2, 4)
plt.plot(range(1, iters_num+1), penalty_list_1, 'o-', label='pulses_spsa', linewidth=2, markersize=8)
plt.xlabel('运行次数')
plt.ylabel('惩罚')
plt.title('惩罚对比')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
np.max(score_list_1)